In [2]:
import numpy as np

numbers = np.random.randint(1, 1000000, size=100000, dtype=np.int32)

np.savetxt("sorting_dataset.txt", numbers, fmt="%d")

In [3]:
import os

print(os.getcwd())

/kaggle/working


In [4]:
import os

print(os.listdir("/kaggle/working"))

['.virtual_documents', 'sorting_dataset.txt']


In [5]:
print("Total numbers:", len(numbers))

Total numbers: 100000


In [6]:
import numpy as np

numbers = np.loadtxt("/kaggle/working/sorting_dataset.txt",
                     dtype=np.int32)

print(numbers[:10])  # First 10 numbers

[852091 893048  54356 354074 758980 986274 230418 959058 610207 394313]


In [7]:
import numpy as np
from numba import cuda
import math

# CUDA Kernel
@cuda.jit
def prime_check_kernel(numbers, prime_flags):
    idx = cuda.grid(1)

    if idx < numbers.size:
        n = numbers[idx]

        if n < 2:
            prime_flags[idx] = 0
            return

        if n == 2:
            prime_flags[idx] = 1
            return

        if n % 2 == 0:
            prime_flags[idx] = 0
            return

        is_prime = 1

        limit = int(math.sqrt(n)) + 1

        for i in range(3, limit, 2):
            if n % i == 0:
                is_prime = 0
                break

        prime_flags[idx] = is_prime


# Load the Sorting Benchmark Dataset
numbers = np.loadtxt(
    "/kaggle/working/sorting_dataset.txt",
    dtype=np.int32
)

# Output array
prime_flags = np.zeros(numbers.size, dtype=np.int32)

# Copy data to GPU
d_numbers = cuda.to_device(numbers)
d_prime_flags = cuda.to_device(prime_flags)

# CUDA configuration
threads_per_block = 256
blocks_per_grid = (numbers.size + threads_per_block - 1) // threads_per_block

# Launch kernel
prime_check_kernel[blocks_per_grid, threads_per_block](
    d_numbers,
    d_prime_flags
)

# Copy results back to CPU
prime_flags = d_prime_flags.copy_to_host()

# Extract prime numbers
primes = numbers[prime_flags == 1]

# Display results
print("Total Numbers in Dataset :", numbers.size)
print("Total Prime Numbers Found :", len(primes))

print("\nFirst 20 Prime Numbers:")
print(primes[:20])

Total Numbers in Dataset : 100000
Total Prime Numbers Found : 7776

First 20 Prime Numbers:
[815623 574261 248609 104243 975523 379033 643081 587677 504181 536407
 833923 284231 257867 431797 181087 707831 305633 283519 442973 466183]
